# Download KodeKloud Course into Google Drive

## Disclaimer
Please read the following disclaimer carefully before using the Downloader CLI Tool.

- This is intended for personal use only. By using this notebook, you agree to use it at your own risk and assume full responsibility for any consequences that may arise from its use. The developers and contributors of this notebook are not responsible for any damages or losses that may occur from its use.

- The use of this tool to download courses is for educational purposes only. You must have the proper authorization or active subscription from the course provider to access the content legally.

- It is strictly prohibited to distribute or share the downloaded content through any means, including but not limited to uploading to file-sharing platforms, torrent sites, or any other form of digital or physical distribution. Doing so is a violation of copyright laws and may result in legal consequences.

## Step 1: Install kodekloud-downloader

Install the updated downloader directly from the fork repository, along with browser support for token exchange.

> **Note**: FFmpeg is already pre-installed on Google Colab runtimes.

In [ ]:
# Install downloader with browser dependencies from fork
!pip install -U "kodekloud-downloader[browser] @ git+https://github.com/NovoG93/kodekloud-downloader.git"

# Install Chromium for Playwright (required if using automated cookie-to-token exchange)
!playwright install --with-deps chromium

## Step 2: Mount Google Drive

Mount your Google Drive to save courses directly into your cloud storage.

> **Make sure this is the same Google account where this notebook is running.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Choose Authentication Method

KodeKloud uses Firebase Authentication. You can authenticate using either **Option A (Direct Token - Recommended)** or **Option B (Cookie File)**.

---

### Option A (Recommended): Direct Firebase Token
This is the fastest and cleanest method in Google Colab since it requires no file uploads or headless browsers.

#### How to get your Firebase Access Token:
1. Sign in to [kodekloud.com](https://learn.kodekloud.com) in your browser.
2. Open Developer Tools (`F12` or right-click -> **Inspect**).
3. In the **Console** tab, run this snippet to copy your token:
   ```javascript
   (() => {
     const req = indexedDB.open("firebaseLocalStorageDb");
     req.onsuccess = () => {
       const db = req.result;
       const tx = db.transaction("firebaseLocalStorage", "readonly");
       tx.objectStore("firebaseLocalStorage").getAll().onsuccess = (e) => {
         const token = e.target.result[0]?.value?.stsTokenManager?.accessToken;
         if (token) { console.log("Copy this token:\n" + token); } else { console.error("Token not found. Are you logged in?"); }
       };
     };
   })();
   ```
4. Run the cells below and paste your token when prompted.

In [ ]:
import getpass

auth_token = getpass.getpass("Paste your KodeKloud Firebase Access Token: ").strip()

In [ ]:
# Download courses to Google Drive using token
!kodekloud dl -t "$auth_token" -o "/content/drive/MyDrive"

---

### Option B: Upload Cookie File

- Sign in to [kodekloud.com](https://learn.kodekloud.com)
- Export cookies using an extension such as [Get cookies.txt LOCALLY](https://chrome.google.com/webstore/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc/related)
- Make sure the exported cookie file contains `_secure-user-session`.
- The downloader will launch headless Chromium to automatically exchange your session cookie for a Firebase ID token.

In [ ]:
from google.colab import files

uploaded = files.upload()
cookie_file_name = list(uploaded.keys())[0]
print(f"Uploaded cookie file: {cookie_file_name}")

In [ ]:
# Download courses to Google Drive using cookie file
!kodekloud dl -c "$cookie_file_name" -o "/content/drive/MyDrive"